In [8]:
import numpy as np
import pandas as pd
import math
import heapq
import matplotlib.pyplot as plt
from decimal import Decimal
import networkx as nx
floaterr = 1.0e-8

In [9]:
opt_a280 = 2586.76964756316
opt_kz9976 = 1061882
opt_xql662 = 2513
opt_mona = 100000

In [10]:
a = pd.read_csv("./datasets/a280.csv")

In [11]:
import numpy as np
import pandas as pd
import math
import heapq
import matplotlib.pyplot as plt
from decimal import Decimal
import networkx as nx
import itertools # Held-Karp 에 필요
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans


floaterr = 1.0e-8

# 최적해 (참고용)
opt_a280 = 2586.76964756316
opt_kz9976 = 1061882
opt_xql662 = 2513

In [12]:

def held_karp_iter(dist_matrix):
    n = len(dist_matrix)
    if n == 0: return 0, []
    if n == 1: return 0, [0]

    C = {}

    for k in range(1, n):
        C[(1 << k, k)] = (dist_matrix[0][k], 0)

    for size in range(2, n):
        for subset_indices in itertools.combinations(range(1, n), size):
            S = 0
            for bit in subset_indices:
                S |= (1 << bit)
            for k in subset_indices:
                prev_S = S & ~(1 << k)
                min_val = float('inf')
                best_prev_node = -1
                for m in subset_indices:
                    if m == k: continue
                    if not (prev_S & (1 << m)):
                        if prev_S == 0 and m == 0:
                           pass
                        else:
                            continue

                    if (prev_S, m) in C:
                        cost, _ = C[(prev_S, m)]
                        if cost + dist_matrix[m][k] < min_val:
                            min_val = cost + dist_matrix[m][k]
                            best_prev_node = m
                if best_prev_node != -1:
                    C[(S, k)] = (min_val, best_prev_node)

    last_S_mask = 0
    for i in range(1,n):
        last_S_mask |= (1 << i)

    min_tour_len = float('inf')
    last_node_of_tour = -1

    if n == 1:
        return 0, [0]
    
    if not C and n > 1:
        if n == 2:
            min_tour_len = dist_matrix[0][1] + dist_matrix[1][0]
            last_node_of_tour = 1
        else:
            return float('inf'), []

    for k in range(1, n):
        if (last_S_mask, k) in C:
            cost, _ = C[(last_S_mask, k)]
            total_cost = cost + dist_matrix[k][0]
            if total_cost < min_tour_len:
                min_tour_len = total_cost
                last_node_of_tour = k
        elif n==2 and k==1:
             cost = dist_matrix[0][1]
             total_cost = cost + dist_matrix[1][0]
             if total_cost < min_tour_len:
                min_tour_len = total_cost
                last_node_of_tour = 1

    if last_node_of_tour == -1:
        if n > 0:
             return float('inf'), []
        else:
             return 0, []

    path = []
    curr_node = last_node_of_tour
    curr_mask = last_S_mask
    while curr_node != 0 :
        path.append(curr_node)
        if curr_mask == 0 or curr_node == -1 : break
        
        if curr_mask == (1 << curr_node):
            prev_node = 0
        elif (curr_mask, curr_node) in C:
             _, prev_node = C[(curr_mask, curr_node)]
        else:
            return float('inf'), []

        curr_mask &= ~(1 << curr_node)
        curr_node = prev_node
        if curr_node == 0 and curr_mask != 0:
            return float('inf'), []

    path.append(0)
    final_path = path[::-1]

    return min_tour_len, final_path



In [13]:
class Solver:
    def __init__(self, points, ends=None, recursion_depth=0):
        self.points = np.array(points)
        self.n_points = len(self.points)
        self.ends = ends
        self.total_distance = 0
        self.recursion_depth = recursion_depth
        self.MAX_RECURSION_DEPTH = 8

        if self.n_points > 0:
            self.distance_matrix = self.cal_dist_matrix(self.points)
        else:
            self.distance_matrix = np.array([[]])

    def cal_dist_matrix(self, points_arr):
        if not isinstance(points_arr, np.ndarray):
            points_arr = np.array(points_arr)
        if points_arr.ndim == 1 and points_arr.size > 0:
             points_arr = points_arr.reshape(1, -1)
        if len(points_arr) == 0:
            return np.array([[]])
        return cdist(points_arr, points_arr, 'euclidean')

    def find_path_dp(self, dist_matrix, start_node_idx, end_node_idx):
        n = len(dist_matrix)
        if n == 0: return 0
        if n == 1: return 0 if start_node_idx == end_node_idx else float('inf')

        dp = [[float('inf')] * n for _ in range(1 << n)]
        dp[1 << start_node_idx][start_node_idx] = 0

        for mask in range(1 << n):
            for i in range(n):
                if not (mask & (1 << i)): continue
                if dp[mask][i] == float('inf'): continue
                for j in range(n):
                    if not (mask & (1 << j)):
                        new_mask = mask | (1 << j)
                        if dp[mask][i] + dist_matrix[i][j] < dp[new_mask][j]:
                            dp[new_mask][j] = dp[mask][i] + dist_matrix[i][j]
        
        final_dist = dp[(1 << n) - 1][end_node_idx]
        return final_dist if final_dist != float('inf') else 0

    def solve(self):
        if self.n_points == 0:
            return 0
        
        if self.recursion_depth > self.MAX_RECURSION_DEPTH:
            if self.ends is not None and len(self.ends) == 2:
                start_idx, end_idx = self.ends
                return self.find_path_dp(self.distance_matrix, start_idx, end_idx)
            else:
                dist, _ = held_karp_iter(self.distance_matrix)
                return dist if dist != float('inf') else 0

        if self.n_points <= 15:
            if self.ends is not None and len(self.ends) == 2:
                start_idx, end_idx = self.ends
                path_dist = self.find_path_dp(self.distance_matrix, start_idx, end_idx)
                return path_dist
            else:
                circuit_dist, _ = held_karp_iter(self.distance_matrix)
                return circuit_dist if circuit_dist != float('inf') else 0
        else:
            n_clusters_target = 15
            n_clusters = min(n_clusters_target, self.n_points -1 if self.n_points > 1 else 1)
            if self.n_points > 1 and n_clusters <=1 :
                 n_clusters = max(2, int(np.sqrt(self.n_points/2)))
                 n_clusters = min(n_clusters, self.n_points -1 if self.n_points > 1 else 1)
            if n_clusters == 0 and self.n_points > 0 : n_clusters = 1
            if n_clusters == 0 and self.n_points == 0 : return 0
            if self.n_points < n_clusters:
                n_clusters = self.n_points

            if n_clusters <= 1:
                if self.ends is not None and len(self.ends) == 2:
                    return self.find_path_dp(self.distance_matrix, self.ends, self.ends)
                else:
                    dist, _ = held_karp_iter(self.distance_matrix)
                    return dist if dist != float('inf') else 0


            kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
            if self.points.shape[0] < n_clusters or self.points.shape[0] == 0:
                if self.ends is not None and len(self.ends) == 2:
                    return self.find_path_dp(self.distance_matrix, self.ends, self.ends)
                else:
                    dist, _ = held_karp_iter(self.distance_matrix)
                    return dist if dist != float('inf') else 0
            else:
                labels = kmeans.fit_predict(self.points)

            cluster_centers = kmeans.cluster_centers_
            clusters_points_list = [self.points[labels == i] for i in range(n_clusters)]

            valid_clusters_indices = [i for i, pts in enumerate(clusters_points_list) if len(pts) > 0]
            if not valid_clusters_indices: return 0
            
            clusters_points_list = [clusters_points_list[i] for i in valid_clusters_indices]
            cluster_centers = cluster_centers[valid_clusters_indices]
            n_clusters = len(clusters_points_list)

            if n_clusters <= 1:
                if self.ends is not None and len(self.ends) == 2:
                    return self.find_path_dp(self.distance_matrix, self.ends, self.ends)
                else:
                    dist, _ = held_karp_iter(self.distance_matrix)
                    return dist if dist != float('inf') else 0

            center_dist_matrix = self.cal_dist_matrix(cluster_centers)
            _, cluster_order_indices = held_karp_iter(center_dist_matrix)

            if not cluster_order_indices or _ == float('inf'):
                if self.ends is not None and len(self.ends) == 2:
                    return self.find_path_dp(self.distance_matrix, self.ends, self.ends)
                else:
                    dist, _ = held_karp_iter(self.distance_matrix)
                    return dist if dist != float('inf') else 0
            
            self.total_distance = 0
            last_actual_exit_coords = None
            first_actual_entry_coords_of_first_cluster = None

            for i in range(len(cluster_order_indices)):
                current_cluster_original_label = cluster_order_indices[i]
                points_in_current_cluster = clusters_points_list[current_cluster_original_label]
                
                if len(points_in_current_cluster) == 0: continue

                entry_idx_in_curr = 0
                if last_actual_exit_coords is not None:
                    dists_to_prev_exit = cdist(points_in_current_cluster, [last_actual_exit_coords])
                    entry_idx_in_curr = np.argmin(dists_to_prev_exit)
                    self.total_distance += dists_to_prev_exit[entry_idx_in_curr, 0]
                
                if i == 0:
                    first_actual_entry_coords_of_first_cluster = points_in_current_cluster[entry_idx_in_curr]
                next_cluster_original_label_in_order = cluster_order_indices[(i + 1) % len(cluster_order_indices)]
                center_of_next_cluster = cluster_centers[next_cluster_original_label_in_order]
                
                exit_idx_in_curr = 0
                if len(points_in_current_cluster) > 0 :
                    dists_to_next_center = cdist(points_in_current_cluster, [center_of_next_cluster])
                    exit_idx_in_curr = np.argmin(dists_to_next_center)

                internal_dist = 0
                if len(points_in_current_cluster) == 1:
                    internal_dist = 0
                elif entry_idx_in_curr == exit_idx_in_curr :
                    sub_solver = Solver(points_in_current_cluster, 
                                            ends=(entry_idx_in_curr, exit_idx_in_curr), 
                                            recursion_depth=self.recursion_depth + 1)
                    internal_dist = sub_solver.solve()
                else:
                    sub_solver = Solver(points_in_current_cluster, 
                                            ends=(entry_idx_in_curr, exit_idx_in_curr), 
                                            recursion_depth=self.recursion_depth + 1)
                    internal_dist = sub_solver.solve()
                
                self.total_distance += internal_dist
                
                if len(points_in_current_cluster) > 0:
                    last_actual_exit_coords = points_in_current_cluster[exit_idx_in_curr]

            if last_actual_exit_coords is not None and first_actual_entry_coords_of_first_cluster is not None:
                self.total_distance += np.linalg.norm(last_actual_exit_coords - first_actual_entry_coords_of_first_cluster)
            
            return self.total_distance


In [15]:
a = pd.read_csv("./datasets/a280.csv")
points_a280 = a[['x', 'y']].values.tolist()
solver_minitsp = Solver(points_a280)
dist_minitsp = solver_minitsp.solve()
print(f"Minitsp (6 points) distance: {dist_minitsp}") # Held-Karp 직접 실행

c:\Users\foxis\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 지정된 파일을 찾을 수 없습니다
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\foxis\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\foxis\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\foxis\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\foxis\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
 

Minitsp (6 points) distance: 2793.7350593098204
